#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json


/n/home07/than157/.conda/envs/llamafactory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load training set of hellaswag
dataset = load_dataset("Rowan/hellaswag", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

print("# samples:", len(dataset)) #should be 39k
n_samples = len(dataset)

#convert to dataframe
df = dataset.to_pandas()
df.head()


Dataset info:
Dataset({
    features: ['ind', 'activity_label', 'ctx_a', 'ctx_b', 'ctx', 'endings', 'source_id', 'split', 'split_type', 'label'],
    num_rows: 39905
})
# samples: 39905


,ind,activity_label,ctx_a,ctx_b,ctx,endings,source_id,split,split_type,label
0,4,Removing ice from car,"Then, the man writes over the snow covering th...",then,"Then, the man writes over the snow covering th...","[, the man adds wax to the windshield and cuts...",activitynet~v_-1IBHYS3L-Y,train,indomain,3
1,8,Baking cookies,A female chef in white uniform shows a stack o...,the pans,A female chef in white uniform shows a stack o...,"[contain egg yolks and baking soda., are then ...",activitynet~v_-2dxp-mv2zo,train,indomain,3
2,9,Baking cookies,A female chef in white uniform shows a stack o...,a knife,A female chef in white uniform shows a stack o...,[is seen moving on a board and cutting out its...,activitynet~v_-2dxp-mv2zo,train,indomain,3
3,12,Baking cookies,A tray of potatoes is loaded into the oven and...,a large tray of meat,A tray of potatoes is loaded into the oven and...,"[is placed onto a baked potato., , ls, and pic...",activitynet~v_-2dxp-mv2zo,train,indomain,3
4,27,Getting a haircut,The man in the center is demonstrating a hairs...,the man in the blue shirt,The man in the center is demonstrating a hairs...,[is standing on the sponge cutting the hair of...,activitynet~v_-JqLjPz-07E,train,indomain,2


In [3]:
### create column with correct answer

#columns
#ctx: starting text, this is a combo of ctx_a and ctx_b -- string
#endings: choices to continue sentence -- numpy array
#label: index of correct answer in endings array -- string

#get correct answer
df['answer'] = df.apply(lambda row: row['endings'][int(row['label'])], axis=1)
df.head()


,ind,activity_label,ctx_a,ctx_b,ctx,endings,source_id,split,split_type,label,answer
0,4,Removing ice from car,"Then, the man writes over the snow covering th...",then,"Then, the man writes over the snow covering th...","[, the man adds wax to the windshield and cuts...",activitynet~v_-1IBHYS3L-Y,train,indomain,3,", the man continues removing the snow on his car."
1,8,Baking cookies,A female chef in white uniform shows a stack o...,the pans,A female chef in white uniform shows a stack o...,"[contain egg yolks and baking soda., are then ...",activitynet~v_-2dxp-mv2zo,train,indomain,3,are filled with pastries and loaded into the o...
2,9,Baking cookies,A female chef in white uniform shows a stack o...,a knife,A female chef in white uniform shows a stack o...,[is seen moving on a board and cutting out its...,activitynet~v_-2dxp-mv2zo,train,indomain,3,is used to cut cylinder shaped dough into rounds.
3,12,Baking cookies,A tray of potatoes is loaded into the oven and...,a large tray of meat,A tray of potatoes is loaded into the oven and...,"[is placed onto a baked potato., , ls, and pic...",activitynet~v_-2dxp-mv2zo,train,indomain,3,is prepared then it is removed from the oven b...
4,27,Getting a haircut,The man in the center is demonstrating a hairs...,the man in the blue shirt,The man in the center is demonstrating a hairs...,[is standing on the sponge cutting the hair of...,activitynet~v_-JqLjPz-07E,train,indomain,2,sits on the chair next to the sink.


In [4]:
### create json file

#format data for sft
data = []

for idx, row in df.iterrows():
    item = {
        "instruction": row["ctx"],
        "input": "",
        "output": row["answer"]
    }
    data.append(item)
    #track progress
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    if idx == (n_samples - 1):
        print(f"Processed {idx} rows (last row)")

#save to JSON file
with open("data/hellaswag.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Complete!")

Processed 0 rows
Processed 10000 rows
Processed 20000 rows
Processed 30000 rows
Processed 39904 rows (last row)
Complete!
